# 01 · Corpus jurídico + dataset de avaliação

**Projeto:** RAG sobre legislação brasileira com avaliação rigorosa.

Este notebook:
1. Baixa 5 leis do site do Planalto (CDC, LGPD, Marco Civil da Internet, Lei do Inquilinato e CLT).
2. Limpa o HTML (remove texto revogado e anotações) e divide em **chunks por artigo**.
3. Gera um **dataset de avaliação** com 3 tipos de pergunta:
   - **sintéticas**: geradas por um LLM a partir de artigos sorteados e filtradas por um segundo LLM ("crítico");
   - **golden**: escritas à mão, em linguagem de cidadão comum;
   - **fora do escopo**: perguntas que os documentos *não* respondem (para medir se o sistema sabe dizer "não sei").

**Configuração no Kaggle:** Accelerator = *GPU T4 x2*, Internet = *On*.
Rode com **Save Version → Save & Run All**; a saída (`/kaggle/working`) vira entrada dos notebooks 02 e 03.

⏱️ Tempo estimado de GPU: ~30–40 min.

In [ ]:
!pip install -q bitsandbytes  # para carregar o modelo crítico em 4 bits

In [ ]:
import os, re, json, random, time, unicodedata, gc
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

IN_KAGGLE = Path("/kaggle").exists()
OUT_DIR = Path("/kaggle/working") if IN_KAGGLE else Path("saida_01")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Configuração ----------------
SEED = 42
GEN_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"   # gera as perguntas sintéticas
CRITIC_MODEL_ID = "Qwen/Qwen3-8B"               # avalia/filtra as perguntas (carregado em 4 bits)
ARTIGOS_POR_LEI = 70        # quantos artigos sortear por lei para gerar perguntas
MAX_CHARS_CHUNK = 1500      # artigos maiores que isso são quebrados em sub-chunks
NOTA_MINIMA_CRITICO = 4     # perguntas com nota < 4 (escala 1–5) são descartadas
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"   # usado só para testar o código fora do Kaggle

if SMOKE_TEST:
    GEN_MODEL_ID = CRITIC_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
    ARTIGOS_POR_LEI = 2

random.seed(SEED)

## 1. Download das leis

In [ ]:
LEIS = {
    "CDC":  ("Código de Defesa do Consumidor (Lei 8.078/1990)",
             "https://www.planalto.gov.br/ccivil_03/leis/l8078compilado.htm"),
    "LGPD": ("Lei Geral de Proteção de Dados (Lei 13.709/2018)",
             "https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2018/lei/l13709compilado.htm"),
    "MCI":  ("Marco Civil da Internet (Lei 12.965/2014)",
             "https://www.planalto.gov.br/ccivil_03/_ato2011-2014/2014/lei/l12965.htm"),
    "LI":   ("Lei do Inquilinato (Lei 8.245/1991)",
             "https://www.planalto.gov.br/ccivil_03/leis/l8245compilado.htm"),
    "CLT":  ("Consolidação das Leis do Trabalho (Decreto-Lei 5.452/1943)",
             "https://www.planalto.gov.br/ccivil_03/decreto-lei/del5452compilado.htm"),
}

# Cabeçalhos de navegador: o firewall do Planalto costuma bloquear clientes "robóticos"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/130.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
    "Referer": "https://www.planalto.gov.br/ccivil_03/leis/",
}
HTML_DIR = OUT_DIR / "html"
HTML_DIR.mkdir(exist_ok=True)


def decodificar(conteudo: bytes) -> str:
    """O Planalto usa windows-1252 na maioria das páginas, mas algumas são UTF-8."""
    try:
        return conteudo.decode("utf-8")
    except UnicodeDecodeError:
        return conteudo.decode("cp1252", errors="replace")


def parece_lei(html: str) -> bool:
    """Detecta páginas de bloqueio ("The requested URL was rejected"), que chegam com status 200."""
    return len(html) > 5_000 and len(re.findall(r"Art\.?\s*\d", html)) >= 10


def html_de_input(sigla: str, url: str) -> str | None:
    """Fallback: HTMLs salvos pelo navegador e enviados como Dataset do Kaggle.
    Aceita o nome da sigla (CDC.html) ou o nome original do site (l8078compilado.htm)."""
    if not IN_KAGGLE:
        return None
    nomes = {f"{sigla}.html", f"{sigla}.htm", url.rsplit("/", 1)[-1].lower()}
    for arquivo in Path("/kaggle/input").rglob("*.htm*"):
        if arquivo.name.lower() in {n.lower() for n in nomes}:
            html = decodificar(arquivo.read_bytes())
            if parece_lei(html):
                print(f"[{sigla}] usando arquivo enviado: {arquivo}")
                return html
    return None


sessao = requests.Session()
sessao.headers.update(HEADERS)


def baixar(sigla: str, url: str, tentativas: int = 4) -> str:
    destino = HTML_DIR / f"{sigla}.html"
    if destino.exists():
        html = destino.read_text(encoding="utf-8")
        if parece_lei(html):
            return html
        destino.unlink()  # cache de uma execução anterior que recebeu a página de bloqueio
    html = html_de_input(sigla, url)
    if html:
        return html
    recebido = ""
    for i in range(tentativas):
        try:
            r = sessao.get(url, timeout=60)
            r.raise_for_status()
            recebido = decodificar(r.content)
            if parece_lei(recebido):
                destino.write_text(recebido, encoding="utf-8")
                return recebido
            print(f"[{sigla}] tentativa {i+1}: a resposta não parece a lei ({len(recebido):,} caracteres)")
        except Exception as e:
            print(f"[{sigla}] tentativa {i+1} falhou: {e}")
        time.sleep(2 ** i)
    trecho = BeautifulSoup(recebido, "lxml").get_text(" ", strip=True)[:300] if recebido else "(nada)"
    raise RuntimeError(
        f"Não consegui baixar a lei {sigla}.\nO site respondeu: {trecho!r}\n\n"
        "Provavelmente o site do Planalto bloqueou o Kaggle. Solução:\n"
        "  1. Abra cada link de LEIS no navegador e use Ctrl+S (Salvar como → 'Página da Web, somente HTML').\n"
        "  2. Crie um Dataset no Kaggle com os 5 arquivos (pode manter os nomes originais, ex. l8078compilado.htm).\n"
        "  3. Adicione o Dataset neste notebook (Add Input) e rode de novo."
    )


htmls = {}
if not SMOKE_TEST:
    for sigla, (_, url) in LEIS.items():
        htmls[sigla] = baixar(sigla, url)
        print(f"{sigla}: {len(htmls[sigla]):,} caracteres de HTML")

## 2. Limpeza e chunking por artigo

Decisões de design (vale citar no README):
- **Texto riscado (`<strike>`) é removido**: nas versões "compiladas" do Planalto ele é texto revogado.
  Indexá-lo faria o RAG responder com a lei antiga.
- **Anotações** como *(Redação dada pela Lei nº ...)* são removidas: só adicionam ruído ao embedding.
- **Chunk = artigo** (com seus parágrafos e incisos). É a unidade natural da lei e permite medir o retrieval
  pelo artigo correto. Artigos muito longos viram sub-chunks que mantêm o mesmo `artigo_id`.

In [ ]:
RE_ARTIGO = re.compile(r"^Art\.?\s*(\d+)\s*(?:º|°|o)?\s*(?:-\s*([A-Z]{1,2}))?\b")
RE_ANOTACAO = re.compile(
    r"\(\s*(?:Redação dada|Incluíd[oa]|Acrescid[oa]|Acrescentad[oa]|Vide|Revogad[oa]|Vigência|Renumerad[oa]|"
    r"Alterad[oa]|Regulamento|Produção de efeitos?|Promulgação|Expressão|Execução suspensa|Inclusão|VETAD[OA])[^()]*\)",
    re.IGNORECASE,
)
RE_ASSINATURA = re.compile(r"^(Brasília|Rio de Janeiro)\s*,\s*\d")
RE_TITULO = re.compile(r"^(TÍTULO|TITULO|CAPÍTULO|CAPITULO|Seção|SEÇÃO|Subseção|SUBSEÇÃO|LIVRO|PARTE)\b")
RE_MARCADOR = re.compile(r"^(Art\.?\s*\d+\S*|§\s*\d+\S*|Parágrafo único\.?|[IVXLC]+\s*[-–]|[a-z]\))")


def linha_vazia(linha: str) -> bool:
    """True para incisos/parágrafos que sobraram vazios depois de remover '(VETADO)' ou '(Revogado)'."""
    return len(RE_MARCADOR.sub("", linha).strip(" .;:-–")) < 3


def normalizar_espacos(t: str) -> str:
    t = t.replace("\xa0", " ").replace("​", "")
    return re.sub(r"\s+", " ", t).strip()


def paragrafos_do_html(html: str) -> list[str]:
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["strike", "s", "del", "script", "style"]):
        tag.decompose()
    for br in soup.find_all("br"):
        br.replace_with("\n")
    linhas = []
    for p in soup.find_all(["p", "h1", "h2", "h3", "h4"]):
        for parte in p.get_text(" ").split("\n"):
            parte = normalizar_espacos(RE_ANOTACAO.sub("", parte))
            if parte:
                linhas.append(parte)
    # Fallback para páginas sem <p>: usa o texto bruto linha a linha
    if sum(bool(RE_ARTIGO.match(l)) for l in linhas) < 5:
        linhas = [normalizar_espacos(RE_ANOTACAO.sub("", l)) for l in soup.get_text("\n").split("\n")]
        linhas = [l for l in linhas if l]
    return linhas


def extrair_artigos(sigla: str, html: str) -> list[dict]:
    artigos, atual, secao, ignorando, apos_titulo = [], None, "", False, False

    def fechar():
        if atual is not None:
            artigos.append(atual)

    for linha in paragrafos_do_html(html):
        m = RE_ARTIGO.match(linha)
        if m:
            fechar()
            ignorando = False
            num = m.group(1) + (f"-{m.group(2)}" if m.group(2) else "")
            # A CLT começa com o decreto que a aprova (Art. 1º e 2º) antes do corpo da lei:
            # quando o Art. 1 reaparece, descartamos o que veio antes.
            if num == "1" and artigos:
                artigos = []
            atual = {"lei": sigla, "artigo": num, "secao": secao, "linhas": [linha]}
        elif RE_ASSINATURA.match(linha):
            fechar()
            atual, ignorando = None, True
        elif RE_TITULO.match(linha) and len(linha) < 120:
            secao, apos_titulo = linha, True
            continue
        elif apos_titulo and linha.isupper() and len(linha) < 150:
            secao = f"{secao} – {linha}"  # nome do capítulo, na linha logo abaixo de "CAPÍTULO VI"
            continue
        elif atual is not None and not ignorando:
            atual["linhas"].append(linha)
        apos_titulo = False
    fechar()

    # Remove incisos vetados/revogados e artigos vazios; trata números repetidos
    vistos, limpos = {}, []
    for a in artigos:
        linhas = [l for i, l in enumerate(a["linhas"]) if i == 0 or not linha_vazia(l)]
        texto = "\n".join(linhas)
        corpo = RE_ARTIGO.sub("", linhas[0]).strip(" .-–") + " ".join(linhas[1:])
        if len(corpo) < 30:
            continue
        vistos[a["artigo"]] = vistos.get(a["artigo"], 0) + 1
        art = a["artigo"] if vistos[a["artigo"]] == 1 else f"{a['artigo']}#{vistos[a['artigo']]}"
        limpos.append({"lei": a["lei"], "artigo": art, "secao": a["secao"], "texto": texto})
    return limpos


def quebrar_em_chunks(artigo: dict, max_chars: int = MAX_CHARS_CHUNK) -> list[str]:
    if len(artigo["texto"]) <= max_chars:
        return [artigo["texto"]]
    linhas = artigo["texto"].split("\n")
    cabecalho = linhas[0]
    partes, buf = [], [cabecalho]
    for l in linhas[1:]:
        if sum(len(x) for x in buf) + len(l) > max_chars and len(buf) > 1:
            partes.append("\n".join(buf))
            buf = [f"(continuação do Art. {artigo['artigo']}) "]
        buf.append(l)
    partes.append("\n".join(buf))
    return partes


def montar_chunks(htmls: dict) -> pd.DataFrame:
    registros = []
    for sigla, html in htmls.items():
        artigos = extrair_artigos(sigla, html)
        print(f"{sigla}: {len(artigos)} artigos vigentes")
        if not artigos:  # diagnóstico: mostra o que o parser enxergou, para descobrir o formato da página
            linhas = paragrafos_do_html(html)
            print(f"  ⚠️ nenhum artigo em {sigla}: {len(html):,} caracteres de HTML, {len(linhas)} linhas de texto, "
                  f"{sum(l.startswith('Art') for l in linhas)} começam com 'Art'. Primeiras linhas:")
            for l in linhas[:15]:
                print("   |", l[:150])
        for a in artigos:
            for j, parte in enumerate(quebrar_em_chunks(a)):
                registros.append({
                    "chunk_id": f"{sigla}:{a['artigo']}:{j}",
                    "artigo_id": f"{sigla}:{a['artigo']}",
                    "lei": sigla,
                    "lei_nome": LEIS[sigla][0],
                    "artigo": a["artigo"],
                    "secao": a["secao"],
                    "texto": parte,
                    # Texto que vai para os índices: o prefixo com lei/seção ajuda o retrieval
                    "texto_indexado": f"{LEIS[sigla][0]} | {a['secao']} | {parte}",
                })
    return pd.DataFrame(registros)


if SMOKE_TEST:  # HTML de exemplo, imitando o formato do Planalto
    htmls = {"CDC": """<html><body>
    <p>Presidência da República</p><p>CAPÍTULO VI</p><p>DA PROTEÇÃO CONTRATUAL</p>
    <p>Art. 49. O consumidor pode desistir do contrato, no prazo de 7 dias a contar de sua assinatura ou do ato de
    recebimento do produto ou serviço, sempre que a contratação ocorrer fora do estabelecimento comercial.</p>
    <p>Parágrafo único. Se o consumidor exercitar o direito de arrependimento, os valores pagos serão devolvidos.</p>
    <p><strike>Art. 50. Texto antigo revogado.</strike></p>
    <p>Art. 50. A garantia contratual é complementar à legal e será conferida mediante termo escrito.
    (Redação dada pela Lei nº 99, de 2099)</p>
    <p>Art. 51. (VETADO)</p>
    <p>Art. 5º-A. Artigo com letra no número para testar o parser de identificadores.</p>
    <p>Brasília, 11 de setembro de 1990; 169º da Independência.</p><p>FERNANDO COLLOR</p>
    </body></html>"""}
    LEIS["CDC"] = ("Código de Defesa do Consumidor (Lei 8.078/1990)", "")

chunks = montar_chunks(htmls)
if chunks.empty:
    raise RuntimeError("Nenhum artigo extraído. Veja o diagnóstico acima e me mande essa saída.")
chunks.to_parquet(OUT_DIR / "chunks.parquet", index=False)
print(f"\nTotal: {len(chunks):,} chunks de {chunks.artigo_id.nunique():,} artigos")
chunks.groupby("lei").agg(artigos=("artigo_id", "nunique"), chunks=("chunk_id", "count"),
                          chars_medio=("texto", lambda s: int(s.str.len().mean())))

In [ ]:
# Inspeção manual: sempre olhe alguns chunks antes de seguir. Erros de parsing contaminam tudo depois.
for _, r in chunks.sample(min(3, len(chunks)), random_state=SEED).iterrows():
    print(f"--- {r.chunk_id} | {r.secao}\n{r.texto[:500]}\n")

## 3. Funções de LLM (geração em lote com `transformers`)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


import transformers
from packaging import version

# transformers >= 4.56 renomeou `torch_dtype` para `dtype`
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"


def carregar_llm(model_id: str, quatro_bits: bool = False):
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # T4 não suporta bfloat16 de forma eficiente → float16
    kwargs = {DTYPE_KW: torch.float16 if DEVICE == "cuda" else torch.float32}
    if quatro_bits and DEVICE == "cuda":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    # Tudo na GPU 0: um modelo de 4B em fp16 (~8 GB) cabe numa T4 de 16 GB
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map={"": 0} if DEVICE == "cuda" else None, **kwargs)
    model.eval()
    return model, tok


@torch.inference_mode()
def gerar(model, tok, prompts: list[str], system: str | None = None, max_new_tokens: int = 300,
          batch_size: int = 8) -> list[str]:
    """Geração gulosa (determinística) em lote. Ordena por tamanho para reduzir padding."""
    textos = [
        tok.apply_chat_template(([{"role": "system", "content": system}] if system else [])
                                + [{"role": "user", "content": p}],
                                tokenize=False, add_generation_prompt=True, enable_thinking=False)
        for p in prompts
    ]
    ordem = sorted(range(len(textos)), key=lambda i: len(textos[i]))
    saidas = [None] * len(textos)
    for i in tqdm(range(0, len(ordem), batch_size), desc="gerando"):
        idx = ordem[i:i + batch_size]
        enc = tok([textos[j] for j in idx], return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
        for j, seq in zip(idx, out[:, enc["input_ids"].shape[1]:]):
            saidas[j] = tok.decode(seq, skip_special_tokens=True).strip()
    return saidas


def extrair_json(texto: str):
    m = re.search(r"\{.*\}", texto, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None


def nota(valor) -> float:
    """Converte a nota do LLM para float em [1, 5]; qualquer outra coisa (texto, bool, null) vira NaN."""
    if isinstance(valor, bool):
        return float("nan")
    try:
        x = float(valor)
    except (TypeError, ValueError):
        return float("nan")
    return x if 1 <= x <= 5 else float("nan")


def liberar():
    """Chame depois de `del modelo` para devolver a memória da GPU."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 4. Perguntas sintéticas

Sorteamos artigos de forma **estratificada por lei** (senão a CLT, que é enorme, dominaria o dataset).
O LLM escreve uma pergunta *como um cidadão comum perguntaria*, sem citar o número do artigo,
para não facilitar artificialmente o BM25.

In [ ]:
PROMPT_GERAR = """Você vai criar uma pergunta para avaliar um sistema de busca em leis brasileiras.

Lei: {lei_nome}
Trecho (Art. {artigo}):
\"\"\"{texto}\"\"\"

Escreva UMA pergunta que um cidadão comum faria e que seja respondida por este trecho.
Regras:
- Use linguagem do dia a dia, como alguém descrevendo a própria situação.
- NÃO cite o número do artigo nem copie frases literais do trecho.
- A pergunta deve fazer sentido sozinha, sem o trecho ao lado.
- A resposta deve ser curta (1 a 3 frases) e totalmente baseada no trecho.

Responda apenas com JSON: {{"pergunta": "...", "resposta": "..."}}"""

artigos = chunks.groupby("artigo_id").agg(lei=("lei", "first"), lei_nome=("lei_nome", "first"),
                                          artigo=("artigo", "first"), texto=("texto", "\n".join)).reset_index()
artigos = artigos[artigos.texto.str.len() > 120]  # artigos muito curtos geram perguntas ruins
amostra = (artigos.sample(frac=1, random_state=SEED).groupby("lei").head(ARTIGOS_POR_LEI)
           .reset_index(drop=True))
print(amostra.lei.value_counts())

gen_model, gen_tok = carregar_llm(GEN_MODEL_ID)
t0 = time.time()
respostas = gerar(gen_model, gen_tok,
                  [PROMPT_GERAR.format(lei_nome=r.lei_nome, artigo=r.artigo, texto=r.texto[:2500])
                   for r in amostra.itertuples()],
                  max_new_tokens=250, batch_size=16)
print(f"Geração: {time.time() - t0:.0f}s")
del gen_model
liberar()

sinteticas = []
for r, resp in zip(amostra.itertuples(), respostas):
    j = extrair_json(resp)
    if j and j.get("pergunta") and j.get("resposta"):
        sinteticas.append({"pergunta": j["pergunta"].strip(), "resposta_referencia": j["resposta"].strip(),
                           "artigo_id": r.artigo_id, "lei": r.lei, "contexto_fonte": r.texto, "origem": "sintetica"})
sinteticas = pd.DataFrame(sinteticas)
print(f"{len(sinteticas)} de {len(amostra)} geraram JSON válido")
sinteticas.head()

## 5. Filtro com LLM crítico

Perguntas sintéticas têm muito lixo (vagas, que dependem do trecho, ou com resposta errada).
Um modelo **maior e diferente do gerador** dá notas de 1 a 5 em três critérios; só mantemos as que tiram
nota ≥ 4 em todos. Essa etapa é a diferença entre um dataset de avaliação confiável e um enviesado.

In [ ]:
PROMPT_CRITICO = """Avalie uma pergunta criada para testar um sistema de busca em leis.

Trecho da lei:
\"\"\"{texto}\"\"\"

Pergunta: {pergunta}
Resposta proposta: {resposta}

Dê notas de 1 a 5:
- "fundamentada": a pergunta pode ser respondida de forma clara e inequívoca usando o trecho?
- "autonoma": a pergunta faz sentido para alguém que NÃO está vendo o trecho? (nota 1 se disser "este artigo", "o trecho" etc.)
- "resposta_correta": a resposta proposta está correta segundo o trecho?

Responda apenas com JSON: {{"fundamentada": n, "autonoma": n, "resposta_correta": n, "justificativa": "..."}}"""

critic_model, critic_tok = carregar_llm(CRITIC_MODEL_ID, quatro_bits=True)
t0 = time.time()
notas_brutas = gerar(critic_model, critic_tok,
                     [PROMPT_CRITICO.format(texto=r.contexto_fonte[:2500], pergunta=r.pergunta,
                                            resposta=r.resposta_referencia) for r in sinteticas.itertuples()],
                     max_new_tokens=150, batch_size=8)
print(f"Crítica: {time.time() - t0:.0f}s")
del critic_model
liberar()

CRITERIOS = ["fundamentada", "autonoma", "resposta_correta"]
notas = [extrair_json(n) or {} for n in notas_brutas]
for c in CRITERIOS:
    sinteticas[f"nota_{c}"] = [nota(n.get(c)) for n in notas]
sinteticas["justificativa_critico"] = [str(n.get("justificativa", "")) for n in notas]

aprovadas = sinteticas[(sinteticas[[f"nota_{c}" for c in CRITERIOS]] >= NOTA_MINIMA_CRITICO).all(axis=1)]
print(f"Aprovadas: {len(aprovadas)} de {len(sinteticas)} ({len(aprovadas) / max(len(sinteticas), 1):.0%})")
print(sinteticas[[f"nota_{c}" for c in CRITERIOS]].describe().round(2))

In [ ]:
# Exemplos reprovados: ajudam a entender (e a explicar no README) por que o filtro existe
for r in sinteticas.drop(aprovadas.index).head(3).itertuples():
    print(f"❌ {r.pergunta}\n   → {r.justificativa_critico}\n")

## 6. Perguntas golden (escritas à mão) e fora do escopo

⚠️ **Revise o artigo esperado de cada pergunta antes de publicar** — confira no texto da lei.
Perguntas golden são a parte mais valiosa da avaliação: medem o sistema com perguntas *reais*,
e não com perguntas no estilo do LLM que gerou as sintéticas.

In [ ]:
GOLDEN = [
    # (pergunta, artigo esperado, resposta de referência)
    ("Comprei um celular pela internet e me arrependi. Posso devolver?", "CDC:49",
     "Sim. Em compras fora do estabelecimento comercial, o consumidor pode desistir em até 7 dias do recebimento, com devolução dos valores pagos."),
    ("Quanto tempo eu tenho para reclamar de um defeito em um produto durável?", "CDC:26",
     "90 dias para produtos e serviços duráveis (30 dias para não duráveis)."),
    ("Meu produto veio com defeito. A loja tem quanto tempo para consertar antes de eu pedir a troca?", "CDC:18",
     "O fornecedor tem até 30 dias para sanar o vício; depois disso o consumidor pode exigir troca, devolução do dinheiro ou abatimento do preço."),
    ("Fui cobrado por algo que não devia e paguei. Tenho direito a receber em dobro?", "CDC:42",
     "Sim, o consumidor cobrado em quantia indevida tem direito à repetição do indébito em dobro, com correção e juros, salvo engano justificável."),
    ("O banco só libera o empréstimo se eu contratar um seguro junto. Isso é permitido?", "CDC:39",
     "Não. Condicionar o fornecimento de um produto ou serviço à compra de outro (venda casada) é prática abusiva."),
    ("Por quanto tempo meu nome pode ficar negativado no cadastro de inadimplentes?", "CDC:43",
     "As informações negativas não podem ficar registradas por período superior a cinco anos."),
    ("Qual a multa máxima por atraso que pode ser cobrada numa compra parcelada?", "CDC:52",
     "A multa de mora não pode ser superior a 2% do valor da prestação."),
    ("A loja anunciou um preço e depois se recusou a vender por esse valor. O que posso fazer?", "CDC:35",
     "O consumidor pode exigir o cumprimento forçado da oferta, aceitar outro produto equivalente ou rescindir o contrato com restituição do valor pago."),
    ("Quais são os direitos que eu tenho sobre os meus dados pessoais em uma empresa?", "LGPD:18",
     "Confirmação do tratamento, acesso, correção, anonimização ou eliminação, portabilidade, informação sobre compartilhamento e revogação do consentimento, entre outros."),
    ("Uma empresa pode usar meus dados sem o meu consentimento?", "LGPD:7",
     "Sim, em hipóteses previstas na lei, como cumprimento de obrigação legal, execução de contrato, proteção ao crédito ou legítimo interesse."),
    ("O que a lei considera como dado pessoal sensível?", "LGPD:5",
     "Dados sobre origem racial ou étnica, convicção religiosa, opinião política, filiação sindical, saúde, vida sexual, dado genético ou biométrico."),
    ("Se uma empresa sofrer um vazamento de dados, ela é obrigada a avisar alguém?", "LGPD:48",
     "Sim. O controlador deve comunicar à ANPD e ao titular a ocorrência de incidente de segurança que possa acarretar risco ou dano relevante."),
    ("Qual o valor máximo da multa que a ANPD pode aplicar a uma empresa?", "LGPD:52",
     "Multa simples de até 2% do faturamento, limitada a R$ 50 milhões por infração."),
    ("Toda empresa precisa ter um encarregado de dados (DPO)?", "LGPD:41",
     "O controlador deve indicar um encarregado pelo tratamento de dados pessoais, cuja identidade e contato devem ser divulgados publicamente."),
    ("Os provedores de internet podem diminuir a velocidade de certos sites ou aplicativos?", "MCI:9",
     "Não. Pelo princípio da neutralidade de rede, os pacotes de dados devem ser tratados de forma isonômica, sem distinção por conteúdo, origem, destino ou aplicação."),
    ("Uma rede social pode ser responsabilizada por um post ofensivo de outro usuário?", "MCI:19",
     "Só se, após ordem judicial específica, não tornar o conteúdo indisponível no prazo assinalado."),
    ("Por quanto tempo os aplicativos precisam guardar os registros de acesso dos usuários?", "MCI:15",
     "Pelo prazo de 6 meses, sob sigilo, em ambiente controlado e de segurança."),
    ("Por quanto tempo a operadora de internet guarda os registros de conexão?", "MCI:13",
     "Pelo prazo de 1 ano, sob sigilo, em ambiente controlado e de segurança."),
    ("Vazaram fotos íntimas minhas. O site precisa tirar do ar sem ordem judicial?", "MCI:21",
     "Sim. O provedor responde subsidiariamente se, após notificação do participante, não remover o conteúdo de nudez ou atos sexuais de caráter privado."),
    ("Quero sair do apartamento alugado antes de acabar o contrato. Preciso pagar multa?", "LI:4",
     "O locatário pode devolver o imóvel pagando a multa pactuada, proporcional ao período de cumprimento do contrato."),
    ("Quais tipos de garantia o dono do imóvel pode exigir no aluguel?", "LI:37",
     "Caução, fiança, seguro de fiança locatícia ou cessão fiduciária de quotas de fundo de investimento; apenas uma modalidade por contrato."),
    ("Qual o valor máximo de caução em dinheiro que pode ser pedido no aluguel?", "LI:38",
     "A caução em dinheiro não pode exceder o equivalente a três meses de aluguel."),
    ("O proprietário vai vender a casa que eu alugo. Tenho preferência na compra?", "LI:27",
     "Sim, o locatário tem preferência para adquirir o imóvel em igualdade de condições com terceiros."),
    ("Quais são as obrigações de quem aluga um imóvel (inquilino)?", "LI:23",
     "Pagar pontualmente o aluguel e encargos, usar o imóvel conforme o convencionado, restituí-lo no estado em que recebeu, entre outras."),
    ("Qual a jornada máxima de trabalho por dia pela CLT?", "CLT:58",
     "A duração normal do trabalho não deve exceder 8 horas diárias."),
    ("Quantas horas extras posso fazer por dia e quanto devo receber a mais?", "CLT:59",
     "Até 2 horas extras diárias, com remuneração pelo menos 50% superior à da hora normal."),
    ("Quem trabalha 8 horas por dia tem direito a quanto tempo de almoço?", "CLT:71",
     "Em trabalho contínuo acima de 6 horas, é obrigatório intervalo de no mínimo 1 hora e, salvo acordo, no máximo 2 horas."),
    ("Minhas férias podem ser divididas em mais de um período?", "CLT:134",
     "Sim, com concordância do empregado, em até três períodos, um deles com no mínimo 14 dias e os demais com no mínimo 5 dias cada."),
    ("Faltei alguns dias no ano. Isso diminui meus dias de férias?", "CLT:130",
     "Sim. São 30 dias com até 5 faltas, 24 dias de 6 a 14 faltas, 18 dias de 15 a 23 faltas e 12 dias de 24 a 32 faltas."),
    ("Quais situações permitem demissão por justa causa?", "CLT:482",
     "Ato de improbidade, incontinência de conduta, desídia, embriaguez habitual, violação de segredo, indisciplina, abandono de emprego, entre outras."),
    ("Fui demitido. Em quantos dias a empresa tem que pagar a rescisão?", "CLT:477",
     "O pagamento das verbas rescisórias deve ser feito em até 10 dias contados do término do contrato."),
    ("Quantos dias dura a licença-maternidade?", "CLT:392",
     "120 dias, sem prejuízo do emprego e do salário."),
    ("Posso faltar no trabalho sem desconto no salário para me casar?", "CLT:473",
     "Sim, até 3 dias consecutivos em virtude de casamento."),
]

FORA_DO_ESCOPO = [
    "Qual a pena para o crime de furto?",
    "Quantos pontos na carteira de motorista levam à suspensão da CNH?",
    "Como funciona a partilha de bens num divórcio?",
    "Qual é a idade mínima para se aposentar pelo INSS?",
    "Quem tem direito à herança quando a pessoa morre sem testamento?",
    "Qual a alíquota do imposto de renda para quem ganha 5 mil reais?",
    "Como faço para tirar o porte de arma de fogo?",
    "Qual o valor da pensão alimentícia que o pai deve pagar?",
    "Quais documentos preciso para adotar uma criança?",
    "Quanto tempo de posse é preciso para pedir usucapião de um terreno?",
    "Qual a pena por dirigir embriagado?",
    "Como funciona o IPVA e quando ele vence?",
    "Quais são as regras para doação de campanha eleitoral?",
    "Qual a multa por desmatar uma área de preservação permanente?",
    "Como um estrangeiro consegue visto de residência no Brasil?",
    "Qual o prazo de validade de um cheque?",
    "Como registrar uma patente de invenção?",
    "Qual o limite de faturamento anual de um MEI?",
    "O que acontece se eu não votar na eleição?",
    "Qual a pena para o crime de estelionato?",
]

golden = pd.DataFrame([{"pergunta": p, "artigo_id": a, "lei": a.split(":")[0], "resposta_referencia": r,
                        "origem": "golden"} for p, a, r in GOLDEN])
fora = pd.DataFrame([{"pergunta": p, "artigo_id": None, "lei": None, "resposta_referencia": None,
                      "origem": "fora_do_escopo"} for p in FORA_DO_ESCOPO])

# Checagem: todo artigo esperado precisa existir no corpus (pega erro de parsing ou de digitação)
faltando = sorted(set(golden.artigo_id) - set(chunks.artigo_id))
print("Artigos golden ausentes do corpus:", faltando or "nenhum ✅")

## 7. Dataset final

In [ ]:
colunas = ["pergunta", "resposta_referencia", "artigo_id", "lei", "origem"]
avaliacao = pd.concat([aprovadas[colunas], golden[colunas], fora[colunas]], ignore_index=True)
avaliacao.insert(0, "qid", [f"q{i:04d}" for i in range(len(avaliacao))])
avaliacao.to_parquet(OUT_DIR / "avaliacao.parquet", index=False)
sinteticas.drop(columns=["contexto_fonte"]).to_csv(OUT_DIR / "sinteticas_com_notas.csv", index=False)

print(avaliacao.groupby(["origem", "lei"], dropna=False).size().unstack(0).fillna(0).astype(int))
print(f"\nSalvo em {OUT_DIR}: chunks.parquet, avaliacao.parquet, sinteticas_com_notas.csv")